In [13]:
import os
import pandas as pd
import yfinance as yf
import FinanceDataReader as fdr
import csv
from pykrx import stock
from datetime import date, timedelta
from pathlib import Path

yf.download는 기본적으로 Date를 인덱스로 두고 OHLCV와 Adj Close를 반환함\
만약 Adj Close가 누락되면 auto_adjust 인자를 다시 False로 지정해 재시도

ks: 코스피, kq: 코스닥

interval : default 일봉, progress : 다운로드 진행 상황 표시 옵션

# 2분봉

In [4]:
# csv 파일 생성 함수
def stock_csv_gen(ticker):
    df = yf.download(
        ticker, period="60d", interval="2m", auto_adjust=False, progress=False)
    # 멀티 인덱스 대응
    if isinstance(df.columns, pd.MultiIndex) and "Ticker" in df.columns.names:
        df.columns = df.columns.droplevel("Ticker")
    df = df.reset_index()
    # Datetime 컬럼명 통일
    if "Datetime" not in df.columns and "Date" in df.columns:
        df = df.rename(columns={"Date": "Datetime"})
    cols = ["Datetime", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    df = df.reindex(columns=cols)
    file_name = f"{ticker}.csv"
    return df, file_name

### ETF

In [6]:
# KOSPI 200 추종 ETF
kospi_tickers = ["069500.KS", "102110.KS"]

for t in kospi_tickers:
    df, file_name = stock_csv_gen(t)
    df.to_csv(file_name, index=False, date_format="%Y-%m-%d %H:%M:%S")

print("생성이 완료되었습니다.")

생성이 완료되었습니다.


In [7]:
# S&P 500 추종 ETF
snp_tickers = ["SPY"]

for t in snp_tickers:
    df, file_name = stock_csv_gen(t)
    df.to_csv(file_name, index=False, date_format="%Y-%m-%d %H:%M:%S")

print("생성이 완료되었습니다.")

생성이 완료되었습니다.


In [8]:
# Nikkei 225 추종 ETF
nikkei_tickers = ["1321.T"]

for t in nikkei_tickers:
    df, file_name = stock_csv_gen(t)
    df.to_csv(file_name, index=False, date_format="%Y-%m-%d %H:%M:%S")

print("생성이 완료되었습니다.")

생성이 완료되었습니다.


In [9]:
# CSI 300 추종 ETF
csi300_tickers = ["510300.SS"]

for t in csi300_tickers:
    df, file_name = stock_csv_gen(t)
    df.to_csv(file_name, index=False, date_format="%Y-%m-%d %H:%M:%S")

print("생성이 완료되었습니다.")

생성이 완료되었습니다.


In [10]:
# EURO STOXX 50 추종 ETF
stoxx50_tickers = ["EUE.DE"]

for t in stoxx50_tickers:
    df, file_name = stock_csv_gen(t)
    df.to_csv(file_name, index=False, date_format="%Y-%m-%d %H:%M:%S")

print("생성이 완료되었습니다.")

생성이 완료되었습니다.


In [ ]:
# STOXX Europe 600 추종 ETF
stoxx600_tickers = ["XSX6.L", "EXSA.DE", "LYP6.DE", "XSX6.SW"]

for t in stoxx600_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    df.to_csv(file_name, index=False)

print("생성이 완료되었습니다.")

LYP6.DE는 데이터가 부족합니다.



1 Failed download:
['XSX6.SW']: YFPricesMissingError('possibly delisted; no price data found  (period=11y) (Yahoo error = "No data found, symbol may be delisted")')


XSX6.SW는 데이터가 부족합니다.
생성이 완료되었습니다.


In [ ]:
# MSCI Europe 추종 ETF
msci_tickers = ["IMAE.AS", "IEUR", "EUNK.DE", "IQQY.DE", "XMEU.DE"]

for t in msci_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    df.to_csv(file_name, index=False)

print("생성이 완료되었습니다.")

IMAE.AS는 데이터가 부족합니다.
EUNK.DE는 데이터가 부족합니다.
생성이 완료되었습니다.


---

In [2]:
# csv 파일 생성 함수
def stock_csv_gen(ticker):
    df = yf.download(
        ticker, period="11y", interval="1d", auto_adjust=False, progress=False)
    df.columns = df.columns.droplevel("Ticker")
    df = df.tail(2500).reset_index()
    cols = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    df = df.reindex(columns=cols)
    file_name = f"{ticker}.csv"
    return df, file_name

In [3]:
# 종목명 조회
def get_name(ticker):
    try:
        yft = yf.Ticker(ticker)
        return yft.info.get("longName") or yft.info.get("shortName") or ticker
    except Exception:
        return ticker

In [4]:
base_dir = Path.cwd()    # 현재 작업 디렉토리

### 국장

#### 코스피

In [3]:
# 가장 최근 영업일의 시장별 종목리스트를 가져옴
krx_stocks = fdr.StockListing('KRX')    # 코스피, 코스닥, 코넥스 전체
krx_stocks

,Code,ISU_CD,Name,Market,Dept,Close,ChangeCode,Changes,ChagesRatio,Open,High,Low,Volume,Amount,Marcap,Stocks,MarketId
0,005930,KR7005930003,삼성전자,KOSPI,,76500,1,1100,1.46,77200,77600,75900,19908805,1529429370850,452852301033000,5919637922,STK
1,000660,KR7000660001,SK하이닉스,KOSPI,,331000,1,2500,0.76,339500,341500,325000,4221256,1404142725000,240968782815000,728002365,STK
2,373220,KR7373220003,LG에너지솔루션,KOSPI,,355500,3,0,0.00,356000,359000,352500,211120,75037044250,83187000000000,234000000,STK
3,207940,KR7207940008,삼성바이오로직스,KOSPI,,1040000,1,2000,0.19,1032000,1042000,1027000,64283,66496943000,74020960000000,71174000,STK
4,012450,KR7012450003,한화에어로스페이스,KOSPI,,986000,2,-16000,-1.60,998000,1008000,980000,143445,142040897000,50841513386000,51563401,STK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2873,245450,KR7245450002,씨앤에스링크,KONEX,일반기업부,1400,1,6,0.43,1400,1500,1400,12,16900,2211944000,1579960,KNX
2874,288490,KR7288490006,나라소프트,KONEX,일반기업부,105,0,0,0.00,0,0,0,0,0,1834515585,17471577,KNX
2875,236030,KR7236030003,씨알푸드,KONEX,일반기업부,898,3,0,0.00,898,898,898,1,898,1825932136,2033332,KNX
2876,308700,KR7308700004,테크엔,KONEX,일반기업부,199,0,0,0.00,0,0,0,0,0,1472600000,7400000,KNX


In [4]:
# 코스피 데이터 리스트
kospi = krx_stocks[krx_stocks["Market"] == "KOSPI"][["Market","Code","Name"]]
kospi["YF"] = kospi["Code"].str.zfill(6) + ".KS"
print(len(kospi))
kospi.head()

960


,Market,Code,Name,YF
0,KOSPI,005930,삼성전자,005930.KS
1,KOSPI,000660,SK하이닉스,000660.KS
2,KOSPI,373220,LG에너지솔루션,373220.KS
3,KOSPI,207940,삼성바이오로직스,207940.KS
4,KOSPI,012450,한화에어로스페이스,012450.KS


In [5]:
kospi_tickers = list(kospi["YF"])

In [ ]:
# KOSPI 폴더 경로
kospi_dir = base_dir / "KOSPI"
kospi_dir.mkdir(parents=True, exist_ok=True)    # 폴더 없으면 생성

In [ ]:
# 코스피 메타 CSV 경로
meta_path = Path("kospi_ticker_names.csv")

# 기존에 기록된 티커 읽어 중복 방지 집합 만들기
seen = set()
if meta_path.exists():
    with meta_path.open("r", encoding="utf-8", newline="") as f:
        r = csv.reader(f)   # CSV 파서로 안전하게 읽기
        for row in r:
            if not row:
                continue
            ticker = row[0].strip()    # 첫 컬럼만 문자열로
            seen.add(ticker)

In [ ]:
# KOSPI 종목 티커별 파일 생성
name_map = {t: get_name(t) for t in kospi_tickers}

for t in kospi_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue

    # 헤더 없는 메타 파일에 Ticker와 Name 한 줄 저장
    # 현재 작업 폴더에 kospi_ticker_names.csv 파일 "추가 모드"로 열고 블록이 끝나면 자동으로 닫음
    if t not in seen:
        with open("kospi_ticker_names.csv", "a", encoding="utf-8", newline="") as f:
            f.write(f"{t},{name_map.get(t, t)}\n")
        seen.add(t)
    
    # 티커별 파일
    df.to_csv(kospi_dir / file_name, index=False)
    
print("생성이 완료되었습니다.")

#### 코스닥

In [6]:
# 코스닥 데이터 리스트
kosdaq = krx_stocks[krx_stocks["Market"] == "KOSDAQ"][["Market","Code","Name"]]
kosdaq["YF"] = kosdaq["Code"].str.zfill(6) + ".KQ"
print(len(kosdaq))
kosdaq.head()

1750


,Market,Code,Name,YF
74,KOSDAQ,087010,펩트론,087010.KQ
89,KOSDAQ,277810,레인보우로보틱스,277810.KQ
91,KOSDAQ,298380,에이비엘바이오,298380.KQ
99,KOSDAQ,028300,HLB,028300.KQ
102,KOSDAQ,000250,삼천당제약,000250.KQ


In [7]:
kosdaq_tickers = list(kosdaq["YF"])

In [ ]:
# KOSDAQ 폴더 경로
kosdaq_dir = base_dir / "KOSDAQ"
kosdaq_dir.mkdir(parents=True, exist_ok=True)    # 폴더 없으면 생성

In [ ]:
# 코스닥 메타 CSV 경로
meta_path = Path("kosdaq_ticker_names.csv")

# 기존에 기록된 티커 읽어 중복 방지 집합 만들기
seen = set()
if meta_path.exists():
    with meta_path.open("r", encoding="utf-8", newline="") as f:
        r = csv.reader(f)   # CSV 파서로 안전하게 읽기
        for row in r:
            if not row:
                continue
            ticker = row[0].strip()    # 첫 컬럼만 문자열로
            seen.add(ticker)

In [ ]:
# KOSDAQ 종목 티커별 파일 생성
name_map = {t: get_name(t) for t in kosdaq_tickers}

for t in kosdaq_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue

    if t not in seen:
        with open("kosdaq_ticker_names.csv", "a", encoding="utf-8", newline="") as f:
            f.write(f"{t},{name_map.get(t, t)}\n")
        seen.add(t)
    
    # 티커별 파일
    df.to_csv(kosdaq_dir / file_name, index=False)
    
print("생성이 완료되었습니다.")

### 미장

#### 나스닥

In [2]:
# 가장 최근 영업일의 시장별 종목리스트를 가져옴
nasdaq_stocks = fdr.StockListing('NASDAQ')  # 나스닥 상장 리스트
nasdaq_stocks

100%|██████████| 3721/3721 [00:08<00:00, 424.88it/s]


,Symbol,Name,IndustryCode,Industry
0,NVDA,NVIDIA Corp,57101010,반도체
1,MSFT,Microsoft Corp,57201020,소프트웨어
2,AAPL,Apple Inc,57106020,전화 및 소형 장치
3,AMZN,Amazon.com Inc,53402010,백화점
4,META,Meta Platforms Inc,57201030,온라인 서비스
...,...,...,...,...
3716,ADAMG,Adamas 9 125 Senior Notes Due 2030,60102040,특수 REITs
3717,SPEGR,Silver Pegasus Acquisition Rights Exp 26 Jun 2030,55601010,투자 지주 회사
3718,MLCI,Mount Logan Capital Inc,,
3719,CHECU,Chenghe Acquisition III Units,55601010,투자 지주 회사


In [3]:
# 나스닥 데이터 리스트
nasdaq = nasdaq_stocks[["Symbol","Name"]]
print(len(nasdaq))
nasdaq.head()

3721


,Symbol,Name
0,NVDA,NVIDIA Corp
1,MSFT,Microsoft Corp
2,AAPL,Apple Inc
3,AMZN,Amazon.com Inc
4,META,Meta Platforms Inc


In [4]:
nasdaq_tickers = list(nasdaq["Symbol"])

In [ ]:
# NASDAQ 폴더 경로
nasdaq_dir = base_dir / "NASDAQ"
nasdaq_dir.mkdir(parents=True, exist_ok=True)    # 폴더 없으면 생성

In [ ]:
# 나스닥 메타 CSV 경로
meta_path = Path("nasdaq_ticker_names.csv")

# 기존에 기록된 티커 읽어 중복 방지 집합 만들기
seen = set()
if meta_path.exists():
    with meta_path.open("r", encoding="utf-8", newline="") as f:
        r = csv.reader(f)   # CSV 파서로 안전하게 읽기
        for row in r:
            if not row:
                continue
            ticker = row[0].strip()    # 첫 컬럼만 문자열로
            seen.add(ticker)

In [ ]:
# NASDAQ 종목 티커별 파일 생성
name_map = {t: get_name(t) for t in nasdaq_tickers}

for t in nasdaq_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue

    if t not in seen:
        with open("nasdaq_ticker_names.csv", "a", encoding="utf-8", newline="") as f:
            f.write(f"{t},{name_map.get(t, t)}\n")
        seen.add(t)
    
    # 티커별 파일
    df.to_csv(nasdaq_dir / file_name, index=False)
    
print("생성이 완료되었습니다.")

#### 뉴욕증권거래소

In [63]:
# 가장 최근 영업일의 시장별 종목리스트를 가져옴
nyse_stocks = fdr.StockListing('NYSE')  # 뉴욕거래소 상장 리스트
nyse_stocks

2723it [00:03, 683.78it/s]                          


,Symbol,Name,IndustryCode,Industry
0,TSM,Taiwan Semiconductor Manufacturing Co Ltd ADR,57101010,반도체
1,JPM,JPMorgan Chase & Co,55101010,은행
2,ORCL,Oracle Corp,57201020,소프트웨어
3,WMT,Walmart Inc,54301020,식품 소매 및 유통
4,LLY,Eli Lilly and Co,56201040,제약
...,...,...,...,...
2718,EMO RT WI,ClearBridge Energy Midstream Opportunity Right...,55501030,폐쇄형 펀드
2719,WBI,WaterBridge Infrastructure LLC,,
2720,EMO RT,ClearBridge Energy Midstream Opportunity Right...,55501030,폐쇄형 펀드
2721,RIV RT,RiverNorth Opportunities Rights Exp 06th Octob...,55501030,폐쇄형 펀드


In [64]:
# 뉴욕증권거래소 데이터 리스트
nyse = nyse_stocks[["Symbol","Name"]]
print(len(nyse))
nyse.head()

2723


,Symbol,Name
0,TSM,Taiwan Semiconductor Manufacturing Co Ltd ADR
1,JPM,JPMorgan Chase & Co
2,ORCL,Oracle Corp
3,WMT,Walmart Inc
4,LLY,Eli Lilly and Co


In [65]:
nyse_tickers = list(nyse["Symbol"])

In [66]:
# NYSE 폴더 경로
nyse_dir = base_dir / "NYSE"
nyse_dir.mkdir(parents=True, exist_ok=True)    # 폴더 없으면 생성

In [67]:
# 뉴욕거래소 메타 CSV 경로
meta_path = Path("nyse_ticker_names.csv")

# 기존에 기록된 티커 읽어 중복 방지 집합 만들기
seen = set()
if meta_path.exists():
    with meta_path.open("r", encoding="utf-8", newline="") as f:
        r = csv.reader(f)   # CSV 파서로 안전하게 읽기
        for row in r:
            if not row:
                continue
            ticker = row[0].strip()    # 첫 컬럼만 문자열로
            seen.add(ticker)

In [ ]:
# NYSE 종목 티커별 파일 생성
name_map = {t: get_name(t) for t in nyse_tickers}

for t in nyse_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue

    if t not in seen:
        with open("nyse_ticker_names.csv", "a", encoding="utf-8", newline="") as f:
            f.write(f"{t},{name_map.get(t, t)}\n")
        seen.add(t)
    
    # 티커별 파일
    df.to_csv(nyse_dir / file_name, index=False)
    
print("생성이 완료되었습니다.")

### 일장

TSE : Tokyo Stock Exchange

In [ ]:
# 가장 최근 영업일의 시장별 종목리스트를 가져옴
tse_stocks = fdr.StockListing('TSE')  # 도쿄증권거래소 상장 리스트
tse_stocks

In [52]:
# tse 데이터 리스트
tse = tse_stocks.loc[:, ["Symbol","Name"]]
tse["YF"] = tse["Symbol"].str.zfill(4) + ".T"
print(len(tse))
tse.head()

4029


,Symbol,Name,YF
0,7203,Toyota Motor Corp,7203.T
1,8306,Mitsubishi UFJ Financial Group Inc,8306.T
2,9984,SoftBank Group Corp,9984.T
3,6758,Sony Group Corp,6758.T
4,6501,Hitachi Ltd,6501.T


In [53]:
tse_tickers = list(tse["YF"])

In [54]:
# TSE 폴더 경로
tse_dir = base_dir / "TSE"
tse_dir.mkdir(parents=True, exist_ok=True)    # 폴더 없으면 생성

In [55]:
# TSE 메타 CSV 경로
meta_path = Path("tse_ticker_names.csv")

# 기존에 기록된 티커 읽어 중복 방지 집합 만들기
seen = set()
if meta_path.exists():
    with meta_path.open("r", encoding="utf-8", newline="") as f:
        r = csv.reader(f)   # CSV 파서로 안전하게 읽기
        for row in r:
            if not row:
                continue
            ticker = row[0].strip()    # 첫 컬럼만 문자열로
            seen.add(ticker)

In [ ]:
# TSE 종목 티커별 파일 생성
name_map = {t: get_name(t) for t in tse_tickers}

for t in tse_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue

    if t not in seen:
        with open("tse_ticker_names.csv", "a", encoding="utf-8", newline="") as f:
            f.write(f"{t},{name_map.get(t, t)}\n")
        seen.add(t)
    
    # 티커별 파일
    df.to_csv(tse_dir / file_name, index=False)
    
print("생성이 완료되었습니다.")

In [48]:
# Nikkei 225 추종 ETF
nikkei_tickers = ["1321.T", "1329.T"]

for t in nikkei_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    df.to_csv(file_name, index=False)

print("생성이 완료되었습니다.")

생성이 완료되었습니다.


In [49]:
# TOPIX 추종 ETF
topix_tickers = ["1306.T"]

for t in topix_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    df.to_csv(file_name, index=False)

print("생성이 완료되었습니다.")

생성이 완료되었습니다.


### 중국장

SSE : Sanghai Stock Exchange

In [57]:
# 가장 최근 영업일의 시장별 종목리스트를 가져옴
sse_stocks = fdr.StockListing('SSE')  # 상하이증권거래소 상장 리스트
sse_stocks

100%|██████████| 1667/1667 [00:01<00:00, 1240.98it/s]


,Symbol,Name,IndustryCode,Industry
0,601288,Agricultural Bank of China Ltd Class A,55101010,은행
1,601398,Industrial and Commercial Bank of China Ltd Cl...,55101010,은행
2,600519,Kweichow Moutai Co Ltd Class A,54101020,증류주 및 포도주
3,601857,PetroChina Co Ltd Class A,50102010,통합 오일 및 가스
4,601988,Bank of China Ltd Class A,55101010,은행
...,...,...,...,...
1662,600696,Shanghai Guijiu Co Ltd Class A,54101020,증류주 및 포도주
1663,600561,Jiangxi Changyun Co Ltd Class A,52406020,지상 및 해상 여객 운송
1664,600193,Shanghai Prosolar Resources Development Co Ltd...,52201020,건설 및 엔지니어링
1665,600321,Rightway Holdings Co Ltd Class A,60101010,"부동산 임대, 개발 및 운영"


In [58]:
# sse 데이터 리스트
sse = sse_stocks.loc[:, ["Symbol","Name"]]
sse["YF"] = sse["Symbol"].str.zfill(6) + ".SS"
print(len(sse))
sse.head()

1667


,Symbol,Name,YF
0,601288,Agricultural Bank of China Ltd Class A,601288.SS
1,601398,Industrial and Commercial Bank of China Ltd Cl...,601398.SS
2,600519,Kweichow Moutai Co Ltd Class A,600519.SS
3,601857,PetroChina Co Ltd Class A,601857.SS
4,601988,Bank of China Ltd Class A,601988.SS


In [59]:
sse_tickers = list(sse["YF"])

In [60]:
# SSE 폴더 경로
sse_dir = base_dir / "SSE"
sse_dir.mkdir(parents=True, exist_ok=True)    # 폴더 없으면 생성

In [61]:
# SSE 메타 CSV 경로
meta_path = Path("sse_ticker_names.csv")

# 기존에 기록된 티커 읽어 중복 방지 집합 만들기
seen = set()
if meta_path.exists():
    with meta_path.open("r", encoding="utf-8", newline="") as f:
        r = csv.reader(f)   # CSV 파서로 안전하게 읽기
        for row in r:
            if not row:
                continue
            ticker = row[0].strip()    # 첫 컬럼만 문자열로
            seen.add(ticker)

In [ ]:
# SSE 종목 티커별 파일 생성
name_map = {t: get_name(t) for t in sse_tickers}

for t in sse_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue

    if t not in seen:
        with open("sse_ticker_names.csv", "a", encoding="utf-8", newline="") as f:
            f.write(f"{t},{name_map.get(t, t)}\n")
        seen.add(t)
    
    # 티커별 파일
    df.to_csv(sse_dir / file_name, index=False)
    
print("생성이 완료되었습니다.")

In [ ]:
# CSI 300 추종 ETF
csi300_tickers = ["510300.SS", "510330.SS", "510310.SS", "510360.SS", "510380.SS"]

for t in csi300_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    df.to_csv(file_name, index=False)

print("생성이 완료되었습니다.")

### 유럽장

In [5]:
# EURO STOXX 50 추종 ETF
stoxx50_tickers = ["EUE.L", "CSX5.L", "VX5E.L", "H50E.L", "XESC.L",
                   "EUE.DE", "XESC.DE", "EL4B.DE",
                   "XESX.SW", "XESC.SW", "E50EUA.SW", "SX5E.SW"]

for t in stoxx50_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    df.to_csv(file_name, index=False)

print("생성이 완료되었습니다.")

VX5E.L는 데이터가 부족합니다.
SX5E.SW는 데이터가 부족합니다.
생성이 완료되었습니다.


In [6]:
# STOXX Europe 600 추종 ETF
stoxx600_tickers = ["XSX6.L", "EXSA.DE", "LYP6.DE", "XSX6.SW"]

for t in stoxx600_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    df.to_csv(file_name, index=False)

print("생성이 완료되었습니다.")

LYP6.DE는 데이터가 부족합니다.



1 Failed download:
['XSX6.SW']: YFPricesMissingError('possibly delisted; no price data found  (period=11y) (Yahoo error = "No data found, symbol may be delisted")')


XSX6.SW는 데이터가 부족합니다.
생성이 완료되었습니다.


In [7]:
# MSCI Europe 추종 ETF
msci_tickers = ["IMAE.AS", "IEUR", "EUNK.DE", "IQQY.DE", "XMEU.DE"]

for t in msci_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    df.to_csv(file_name, index=False)

print("생성이 완료되었습니다.")

IMAE.AS는 데이터가 부족합니다.
EUNK.DE는 데이터가 부족합니다.
생성이 완료되었습니다.


#### 런던 LSE

In [14]:
lse = pd.read_csv("Symbols_LSE.csv")
lse

,Symbol,Description
0,!EXUK.L,EPRA Europe Ex UK Index
1,!G13.L,FTSE Europe LMS Ex Eurobloc Index
2,!SD11.L,FTSE Dev Small Cap Ex US Index
3,01OK.L,Oesterreichische Kontrollbank AG
4,01WE.L,Arab Republic Of Egypt (The)
...,...,...
14087,ZZ70.L,Westpac Securities Nz Limited
14088,ZZ84.L,The Ministry Of Finance Of The People's Republ...
14089,ZZ86.L,The Republic Of Kazakhstan
14090,ZZ87.L,The Ministry Of Finance Of The People's Republ...


In [16]:
lse_tickers = list(lse["Symbol"])

In [ ]:
# LSE 폴더 경로
lse_dir = base_dir / "LSE"
lse_dir.mkdir(parents=True, exist_ok=True)    # 폴더 없으면 생성

In [ ]:
# LSE 종목 티커별 파일 생성
for t in lse_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    
    # 티커별 파일
    df.to_csv(lse_dir / file_name, index=False)
    
print("생성이 완료되었습니다.")

#### 프랑크푸르트 FRA

In [5]:
fra = pd.read_csv("Symbols_FRA.csv")
fra

,Symbol,Description
0,008.F,Sveafastigheter AB
1,00B.F,AQUAPORIN A/S DK 1
2,00D0.F,SOUTH MANGAN.INV. HD-05
3,00W.F,Aspermont Ltd
4,00XJ.F,ETFS EUR Dly Hdgd Agri DJ-UBS AD ETC
...,...,...
15071,ZYY.F,ASCENCIO SCA
15072,ZZA.F,Cinemark Holdings Inc
15073,ZZF.F,US AUTO PARTS NTWRK - Frankfurt Stock Exchang
15074,ZZG.F,INFORMATION SVC GRP - Frankfurt Stock Exchang


In [6]:
fra_tickers = list(fra["Symbol"])

In [7]:
# FRA 폴더 경로
fra_dir = base_dir / "FRA"
fra_dir.mkdir(parents=True, exist_ok=True)    # 폴더 없으면 생성

In [ ]:
# FRA 종목 티커별 파일 생성
for t in fra_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    
    # 티커별 파일
    df.to_csv(fra_dir / file_name, index=False)
    
print("생성이 완료되었습니다.")

#### 파리 PAR

In [28]:
par = pd.read_csv("Symbols_PAR.csv")
par

,Symbol,Description
0,0P00000F24.PA,AXA Tresor Court Terme C
1,100H.PA,Amundi FTSE 100 UCITS ETF EUR Hedged Acc
2,1053P.PA,BCP S&P500 EUR
3,28IY.PA,iShares iBonds Dec 2028 Term EUR Italy Governm...
4,500.PA,Amundi Index Solutions - Amundi S&P 500 UCITS ...
...,...,...
1160,XBTI.PA,DDA Physical Bitcoin ETP A EUR
1161,XFAB.PA,X Fab Silicon Foundries EV
1162,XIL.PA,Xilam Animation
1163,YIEL.PA,Lyxor UCITS iBoxx EUR Liquid High Yield 30 Ex-...


In [29]:
par_tickers = list(par["Symbol"])

In [30]:
# PAR 폴더 경로
par_dir = base_dir / "PAR"
par_dir.mkdir(parents=True, exist_ok=True)    # 폴더 없으면 생성

In [ ]:
# PAR 종목 티커별 파일 생성
for t in par_tickers:
    df, file_name = stock_csv_gen(t)
    if len(df) < 2500:
        print(f"{t}는 데이터가 부족합니다.")
        continue
    
    # 티커별 파일
    df.to_csv(par_dir / file_name, index=False)
    
print("생성이 완료되었습니다.")